# Convolutional Neural Network

### Importing the libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
tf.__version__

## Part 1 - Data Preprocessing

### Preprocessing the Training set

In [ ]:
train_datagen = ImageDataGenerator(rescale = 1./255,shear_range = 0.2,zoom_range = 0.2,horizontal_flip = True)

training_set = train_datagen.flow_from_directory('dataset/training_set',
                                                 target_size = (64, 64),
                                                 batch_size = 32,
                                                 class_mode = 'binary')

### Preprocessing the Test set

In [ ]:
test_datagen = ImageDataGenerator(rescale = 1./255)
test_set = test_datagen.flow_from_directory('dataset/test_set',
                                            target_size = (64, 64),
                                            batch_size = 32,
                                            class_mode = 'binary')

## Part 2 - Building the CNN

### Initialising the CNN

In [ ]:
cnn = tf.keras.models.Sequential()

### Step 1 - Convolution

In [ ]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[64, 64, 3]))

### Step 2 - Pooling

In [ ]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Adding a second convolutional layer

In [ ]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Step 3 - Flattening

In [ ]:
cnn.add(tf.keras.layers.Flatten())

### Step 4 - Full Connection

In [ ]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

### Step 5 - Output Layer

In [ ]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

## Part 3 - Training the CNN

### Compiling the CNN

In [ ]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

### Training the CNN on the Training set and evaluating it on the Test set

In [ ]:
cnn.fit(x = training_set, validation_data = test_set, epochs = 25)

## Part 4 - Making a single prediction

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
test_image = image.load_img('dataset/single_prediction/cat_or_dog_1.jpg', target_size = (64, 64))
test_image = image.img_to_array(test_image)
test_image = np.expand_dims(test_image, axis = 0)
result = cnn.predict(test_image)
training_set.class_indices
if result[0][0] == 1:
  prediction = 'dog'
else:
  prediction = 'cat'

In [ ]:
print(prediction)

## Image Preprocessing and Data Augmentation

Before training a CNN, the images must be loaded in a consistent format and transformed into values the network can process effectively. In this notebook, preprocessing has two parts:

1. rescale both the training and evaluation images; and
2. apply random data augmentation only to the training images.

### Why augment the training set?

A model overfits when it learns the training examples too closely and fails to generalize to unseen data. One common warning sign is very high training accuracy accompanied by substantially lower validation or test accuracy.

Image augmentation reduces this risk by showing the network plausible variations of each training image. Instead of repeatedly seeing exactly the same pixels, the model sees randomly transformed versions and is encouraged to learn features that remain useful under small visual changes. This increases the effective diversity of the training data without requiring new labeled images.

The legacy generator in this notebook creates transformed images in memory as batches are requested; it does not normally save a permanently enlarged dataset to disk. Augmentation is helpful, but it does not guarantee that overfitting will disappear. Model capacity, regularization, data quality, and dataset size also matter.

### Transformations used in the notebook

```python
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
)
```

- `shear_range` applies a shear: an affine transformation that slants the image. A shear is not the same as a translation.
- `zoom_range` randomly zooms images in or out within the configured range.
- `horizontal_flip=True` randomly reflects images from left to right.
- `rescale=1.0 / 255` converts typical 8-bit pixel values from `[0, 255]` to `[0, 1]`.

Augmentations must preserve the meaning of the label. A horizontal flip is usually reasonable for cats and dogs, but it may be invalid when direction matters, such as reading text, recognizing left-versus-right road signs, or identifying laterality in medical images. Transformations should therefore be chosen for the problem rather than copied mechanically.

### Loading the training images

```python
training_set = train_datagen.flow_from_directory(
    'dataset/training_set',
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary',
)
```

`flow_from_directory` expects one subdirectory per class and infers labels from those directory names. Its main arguments here are:

- `target_size=(64, 64)`: resize every image to 64 by 64 pixels so that a batch has a consistent shape. Smaller images train faster but may discard useful detail.
- `batch_size=32`: provide 32 images at a time. This is a common starting point, not a universally optimal setting.
- `class_mode='binary'`: generate binary labels for a two-class task. This matches a model with one sigmoid output and binary cross-entropy loss.

Directory names are normally mapped to integer labels in alphanumeric order. The mapping should be checked rather than assumed:

```python
training_set.class_indices
```

### Preparing evaluation images

```python
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

test_set = test_datagen.flow_from_directory(
    'dataset/test_set',
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary',
)
```

Random augmentation is omitted from the evaluation pipeline because validation and test data should represent the unmodified examples on which the model must generalize. Deterministic preprocessing is still required: evaluation images must be resized and rescaled exactly as the training images are.

Dividing by the fixed constant 255 does not learn anything from the test set, so it does not itself cause data leakage. The familiar `fit_transform` versus `transform` distinction matters when preprocessing estimates quantities from training data, such as a mean and standard deviation. Here, the key requirement is simply to apply the same fixed input convention everywhere.

The notebook passes `test_set` as `validation_data` during training. Because its performance is observed after every epoch and may influence model choices, it is functioning as a **validation set**. For a strict final evaluation, keep a separate test set untouched until architecture and hyperparameter decisions are complete.

### Course API and modern Keras

This notebook uses the course-era `ImageDataGenerator` and `flow_from_directory` workflow. Current Keras code commonly loads directory data with `keras.utils.image_dataset_from_directory` and performs preprocessing with Keras layers. A modern equivalent is:

```python
import keras
from keras import layers

training_set = keras.utils.image_dataset_from_directory(
    'dataset/training_set',
    image_size=(64, 64),
    batch_size=32,
    label_mode='binary',
)

test_set = keras.utils.image_dataset_from_directory(
    'dataset/test_set',
    image_size=(64, 64),
    batch_size=32,
    label_mode='binary',
    shuffle=False,
)

data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomZoom(0.2),
])

cnn = keras.Sequential([
    layers.Input(shape=(64, 64, 3)),
    layers.Rescaling(1.0 / 255),
    data_augmentation,
    # Convolutional and classification layers follow.
])
```

When random augmentation layers are placed inside a Keras model, they apply during training and are inactive during inference. The deterministic `Rescaling` layer applies during both training and inference. The original notebook remains useful for learning the generator workflow, while this alternative reflects the current Keras data-loading style.


## Study Notes: Image Preprocessing

### Pipeline to remember

**Training:** load image -> resize -> rescale -> randomly augment -> batch -> CNN

**Validation/test:** load image -> resize -> rescale -> batch -> CNN

Only label-preserving random augmentation belongs in the training path. Deterministic transformations required by the model belong in every path, including single-image prediction.

### Key distinctions

| Concept | Meaning |
| --- | --- |
| Preprocessing | Deterministic preparation such as resizing or rescaling |
| Data augmentation | Random, label-preserving variation used during training |
| Batch | A group of examples processed before a parameter update |
| Epoch | One complete pass through the training dataset |
| Validation set | Data checked during model development |
| Test set | Data reserved for final, unbiased evaluation |
| Overfitting | Strong training performance but weak unseen-data performance |

### Parameter reference

| Setting | Purpose |
| --- | --- |
| `rescale=1./255` | Maps 8-bit image values from `[0, 255]` to `[0, 1]` |
| `target_size=(64, 64)` | Gives every image the spatial dimensions expected by the CNN |
| `batch_size=32` | Supplies 32 images per batch |
| `class_mode='binary'` | Produces labels for a legacy two-class generator |
| `label_mode='binary'` | Modern directory-dataset equivalent for binary labels |
| `horizontal_flip=True` | Adds random left-right reflections |
| `zoom_range=0.2` | Adds random zoom variation |
| `shear_range=0.2` | Adds random affine shearing in the legacy generator |

### Practical checks

- Confirm that each class has its own correctly named subdirectory.
- Inspect several augmented images to ensure that labels still make sense.
- Verify the inferred class-to-index mapping.
- Use the same image size and deterministic scaling during training, evaluation, and prediction.
- Do not choose augmentation ranges so aggressive that objects become unrealistic or class information disappears.
- Monitor both training and validation curves rather than relying on training accuracy alone.
- Keep a separate final test set when an unbiased performance estimate is required.
- Set evaluation-data shuffling to `False` when predictions must align with filenames or labels in a fixed order.

### Quick self-check

1. Why is random augmentation applied only to training data?
2. Why must evaluation images still be resized and rescaled?
3. Does `ImageDataGenerator` necessarily create new files on disk?
4. What model output and loss match binary labels?
5. Why might horizontal flipping be harmful for some datasets?
6. What is the trade-off when reducing images from 150 by 150 to 64 by 64?
7. Why is a dataset used as `validation_data` no longer a pristine final test set?

### Answers

1. Augmentation regularizes learning by exposing the model to varied training examples; evaluation should measure performance on the natural, unmodified data distribution.
2. The model requires the same shape and numerical input convention it saw during training.
3. No. In this workflow, augmented variants are normally generated in memory as batches are requested.
4. A single sigmoid output with binary cross-entropy.
5. A flip may change the label or remove task-relevant directional meaning.
6. Smaller images reduce computation and training time but can remove fine visual detail.
7. Repeatedly observing its results can influence model decisions, indirectly fitting the development process to that dataset.

### Final takeaway

Resizing and rescaling make inputs consistent; augmentation makes the training data more varied. A sound pipeline augments only training images, applies identical deterministic preprocessing everywhere, and preserves truly unseen data for final evaluation.


## Building the CNN Architecture

The second part of the implementation constructs the complete convolutional neural network. A CNN is still a neural network composed of layers; its distinguishing feature is that convolutional layers learn spatial patterns before dense layers perform the final classification.

### Initializing a Sequential model

The course creates an empty `Sequential` model and adds one layer at a time:

```python
cnn = tf.keras.models.Sequential()
```

`Sequential` is appropriate when every layer has one input and one output and the layers form a straight stack. It is not the opposite of a computational graph; Keras still builds a graph of operations. For architectures with branches, skip connections, or multiple inputs or outputs, the Functional API is more suitable.

Current Keras style often declares the input explicitly:

```python
cnn = tf.keras.Sequential([
    tf.keras.Input(shape=(64, 64, 3)),
])
```

The batch dimension is omitted from `shape`. With the default channels-last format, `(64, 64, 3)` means height 64, width 64, and three RGB channels. A grayscale image normally has one channel and therefore uses `(64, 64, 1)`.

### Step 1: First convolutional layer

```python
cnn.add(
    tf.keras.layers.Conv2D(
        filters=32,
        kernel_size=3,
        activation='relu',
        input_shape=(64, 64, 3),
    )
)
```

The important arguments are:

- `filters=32`: produce 32 output feature maps. Each filter learns to respond to a different pattern.
- `kernel_size=3`: use a 3 by 3 spatial kernel. The shorthand `3` is equivalent to `(3, 3)`.
- `activation='relu'`: apply the Rectified Linear Unit after convolution to introduce non-linearity.
- `input_shape=(64, 64, 3)`: specify the shape of one input image. This is needed only at the model input, not on every layer.

A filter in the first layer is not merely a flat 3 by 3 matrix. It spans all input channels, so each kernel has shape 3 by 3 by 3. In the second convolutional layer, each kernel spans all 32 incoming feature maps and has shape 3 by 3 by 32.

The default convolution stride is 1 and the default padding is `'valid'`. Therefore, a 3 by 3 convolution reduces each 64 by 64 spatial dimension to 62 by 62. Using `padding='same'` with stride 1 would preserve the spatial dimensions.

### Step 2: Max pooling

```python
cnn.add(
    tf.keras.layers.MaxPool2D(
        pool_size=2,
        strides=2,
    )
)
```

Max pooling takes the largest activation in each 2 by 2 window and moves the window by two pixels at a time. This downsamples the feature maps, reduces computation, and provides limited robustness to small spatial changes. It has no trainable weights.

`pool_size=2` is shorthand for `(2, 2)`. If `strides` is omitted, Keras defaults it to `pool_size`, so `MaxPool2D(pool_size=2)` would behave the same here.

With `'valid'` padding, only complete pooling windows are used. With `'same'` padding, Keras pads as needed, but a stride larger than 1 can still reduce the output size; `'same'` does not always mean that pooling preserves the original dimensions.

### Adding a second convolutional block

```python
cnn.add(tf.keras.layers.Conv2D(32, 3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))
```

The second convolutional layer receives the first block's feature maps rather than the original image. It can therefore combine simple patterns into more complex representations. The `input_shape` argument is omitted because Keras infers this layer's input from the preceding layer.

Using 32 filters in both blocks is a design choice, not a fixed CNN rule. The numbers of filters, kernel sizes, and number of blocks are hyperparameters that should be selected using validation performance and computational constraints.

### Step 3: Flattening

```python
cnn.add(tf.keras.layers.Flatten())
```

`Flatten` reshapes the final stack of feature maps into one vector per image. It does not learn any parameters and does not mix values; it only changes their arrangement so they can enter a dense layer.

### Step 4: Full connection

```python
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))
```

This dense layer learns combinations of the extracted visual features. `units=128` means that it contains 128 neurons. The number 128 is a tunable architectural choice rather than a guarantee of better accuracy. More neurons increase model capacity and computation and can also increase overfitting risk.

### Step 5: Binary output layer

```python
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))
```

For mutually exclusive binary classification, one sigmoid neuron produces a value between 0 and 1. This value is interpreted as the probability of whichever class was encoded as 1. The output should be paired with binary cross-entropy during compilation. The class mapping can be confirmed with `training_set.class_indices`.

For mutually exclusive multiclass classification, the usual design is one output neuron per class, a softmax activation, and categorical or sparse categorical cross-entropy. For multilabel classification, use one independent sigmoid output per label.

### Complete architecture

The course model can be expressed compactly as:

```python
cnn = tf.keras.Sequential([
    tf.keras.Input(shape=(64, 64, 3)),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=2, strides=2),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=2, strides=2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])
```

The existing notebook's `tf.keras` syntax remains consistent with its TensorFlow-based course code. Standalone Keras 3 code can use `import keras` and the same layer names through `keras.layers`.


## Study Notes: CNN Architecture

### Architecture flow

**Input -> Conv2D -> ReLU -> MaxPool -> Conv2D -> ReLU -> MaxPool -> Flatten -> Dense -> Sigmoid**

The convolutional blocks extract spatial features, while the dense layers use those features to make the final prediction.

### Shape trace for this model

Because both convolutions use the defaults `strides=1` and `padding='valid'`, and both pooling layers use a 2 by 2 window with stride 2, the tensor shapes are:

| Stage | Output shape per image | Trainable parameters |
| --- | ---: | ---: |
| Input | 64 x 64 x 3 | 0 |
| Conv2D: 32 filters, 3 x 3 | 62 x 62 x 32 | 896 |
| MaxPool2D: 2 x 2 | 31 x 31 x 32 | 0 |
| Conv2D: 32 filters, 3 x 3 | 29 x 29 x 32 | 9,248 |
| MaxPool2D: 2 x 2 | 14 x 14 x 32 | 0 |
| Flatten | 6,272 | 0 |
| Dense: 128 units | 128 | 802,944 |
| Dense: 1 unit | 1 | 129 |

**Total trainable parameters: 813,217.**

The parameter formulas are:

- convolution: `(kernel height x kernel width x input channels + 1 bias) x filters`;
- dense: `(input values x units) + one bias per unit`.

The 128-unit dense layer contains most of this model's parameters. This is why flattening into a large dense layer can make a CNN parameter-heavy. Global average pooling is a common alternative in modern architectures.

Run the following after constructing the network to verify every shape and parameter count:

```python
cnn.summary()
```

### Layer reference

| Layer | Purpose | Learns parameters? |
| --- | --- | --- |
| `Input` | Declares the expected sample shape | No |
| `Conv2D` | Learns local spatial filters | Yes |
| ReLU | Introduces non-linearity | No |
| `MaxPooling2D` | Downsamples each feature map | No |
| `Flatten` | Reshapes feature maps into a vector | No |
| `Dense` | Learns global feature combinations | Yes |
| Sigmoid | Maps one logit to a binary probability | No |

### Important corrections and nuances

- The library is **Keras**, and `Conv2D` means two-dimensional convolution.
- A stride describes how far a filter or pooling window moves; it is not called a slide.
- `padding='valid'` means no padding, not that discarded border information is invalid.
- `padding='same'` preserves convolution dimensions only when stride is 1.
- Pooling does not learn parameters and does not guarantee invariance or prevent overfitting.
- ReLU is a strong default for hidden layers, but it is not mandatory for every architecture.
- More filters or dense units do not automatically produce a better model.
- The class encoded as 1 determines what the sigmoid probability represents.

### Quick self-check

1. Why is `Sequential` suitable for this architecture?
2. What does `filters=32` mean in a `Conv2D` layer?
3. Why does the first convolutional kernel span three channels?
4. Why is `input_shape` specified only at the model input?
5. Does max pooling contain trainable weights?
6. What is the flattened vector length after the second pooling layer?
7. Why does the dense layer dominate the parameter count?
8. Which output activation and loss fit this binary classifier?

### Answers

1. The model is a single linear stack in which each layer has one input and one output.
2. The layer learns 32 filters and produces 32 output feature maps.
3. Each filter must combine information from the red, green, and blue input channels.
4. Later input shapes are inferred automatically from the preceding layers.
5. No. It computes local maxima using fixed window and stride settings.
6. `14 x 14 x 32 = 6,272` values.
7. Every one of the 6,272 flattened values connects to all 128 dense neurons.
8. One sigmoid output paired with binary cross-entropy.

### Final takeaway

This CNN progressively turns pixels into feature maps, reduces their spatial size, converts the learned representation into a vector, and produces a binary probability. Understanding the tensor shape and parameter count after each layer makes the architecture far easier to design, debug, and improve.


## Training the CNN

The architecture defines what the CNN can compute; training determines the parameter values that make those computations useful. During training, the convolutional filters and dense-layer weights are adjusted so that the model's predictions increasingly agree with the known labels.

Training has two main steps in Keras:

1. **compile the model** by selecting an optimizer, loss function, and evaluation metrics; and
2. **fit the model** to batches of training images for a chosen number of epochs.

### Step 1: Compiling the CNN

The notebook compiles the model as follows:

```python
cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'],
)
```

Compiling configures the training process; it does not train the model or update any weights.

#### Optimizer: Adam

`'adam'` selects the **Adam** optimizer—not “Atom.” Adam is a gradient-based optimizer that maintains adaptive estimates of first and second moments of the gradients. With mini-batches, it performs stochastic parameter updates, but it is more specific than plain stochastic gradient descent.

At a high level, each update follows this cycle:

1. run a forward pass on a batch;
2. calculate the batch loss;
3. use backpropagation to compute gradients; and
4. let Adam use those gradients to update the trainable parameters.

Passing the string `'adam'` uses Keras's default Adam configuration. When control over settings is required, instantiate it explicitly:

```python
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
```

The learning rate is a major hyperparameter: one that is too large can make training unstable, while one that is too small can make progress very slow.

#### Loss: binary cross-entropy

`'binary_crossentropy'` matches this model's single sigmoid output and binary labels. For a target $y$ and predicted probability $p$, the loss for one example is:

$$
L = -[y\log(p) + (1-y)\log(1-p)]
$$

The optimizer minimizes this differentiable quantity. Confidently incorrect predictions receive a large penalty, while confident correct predictions receive a small loss.

#### Metric: accuracy

`metrics=['accuracy']` asks Keras to report the proportion of correctly classified images during training and validation. Accuracy is easy to interpret, but it is not the quantity being minimized; the model optimizes binary cross-entropy.

Accuracy is useful when classes are reasonably balanced and error types have similar costs. For imbalanced or high-stakes tasks, precision, recall, F1 score, ROC-AUC, PR-AUC, sensitivity, or specificity may be more informative.

### Step 2: Fitting the model

The course trains the CNN with:

```python
history = cnn.fit(
    x=training_set,
    validation_data=test_set,
    epochs=25,
)
```

The existing notebook does not assign the return value, but saving it as `history` is useful because it contains the loss and metric values recorded after every epoch.

#### `x=training_set`

`training_set` is the batch-producing directory iterator created during preprocessing. It supplies resized, rescaled, and randomly augmented image batches together with their labels. Because the generator already uses `batch_size=32`, no new batch size needs to be passed to `fit()`.

The model updates its weights after processing each training batch. One epoch finishes after it has worked through the full training iterator once. Since augmentation is generated dynamically, the precise transformed versions can differ in later epochs.

#### `validation_data=test_set`

At the end of every epoch, Keras evaluates the current model on `test_set` and reports values such as `val_loss` and `val_accuracy`. Validation performs forward passes only: it does not backpropagate the validation loss or update model weights.

Training and validation within the same `fit()` call is standard supervised-learning behavior and is not specific to computer vision. The two roles remain separate even though Keras displays their results together.

Because this dataset is inspected after every epoch and may influence the epoch count or other design choices, it is functioning as a **validation set**, despite being named `test_set` in the course. A rigorous workflow reserves a third, untouched test set for one final evaluation after all model selection is complete.

#### `epochs=25`

An epoch is one complete pass through the training dataset. Setting `epochs=25` specifies a fixed maximum of 25 passes; it does not guarantee convergence or optimal generalization. The appropriate number depends on the data, architecture, optimizer, learning rate, augmentation, and hardware.

Watch both training and validation curves:

- if both losses remain high, the model may be underfitting or learning too slowly;
- if training loss falls while validation loss rises, the model is probably overfitting; and
- if validation loss stops improving, additional epochs may provide little benefit.

### A safer stopping and checkpointing pattern

Instead of deciding in advance that exactly 25 epochs are best, a modern workflow can allow a larger maximum and stop when validation performance stops improving:

```python
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'best_cnn.keras',
        monitor='val_loss',
        save_best_only=True,
    ),
]

history = cnn.fit(
    training_set,
    validation_data=test_set,
    epochs=50,
    callbacks=callbacks,
)
```

`EarlyStopping` limits unnecessary training and can restore the weights from the best validation epoch. `ModelCheckpoint` saves the best model so it can be evaluated or deployed later. These are recommended improvements, not changes to the notebook's existing executable training cell.

### Inspecting the learning curves

If the result of `fit()` is stored, the recorded values are available in `history.history`:

```python
history.history.keys()
# Common keys: loss, accuracy, val_loss, val_accuracy
```

Plotting these values against epoch number is more informative than looking only at the final accuracy. The validation loss is often especially useful for selecting a checkpoint because it incorporates the confidence of the predictions, whereas accuracy records only whether the chosen class was correct.


## Study Notes: Training and Validation

### Training loop at a glance

**For each training batch:**

$$
\text{Images} \rightarrow \text{Forward pass} \rightarrow \text{Loss} \rightarrow \text{Backpropagation} \rightarrow \text{Adam update}
$$

**At the end of each epoch:**

$$
\text{Validation images} \rightarrow \text{Forward pass only} \rightarrow \text{val_loss and val_accuracy}
$$

The validation stage measures generalization but does not train the network.

### Compile versus fit

| Method | Purpose | Updates weights? |
| --- | --- | --- |
| `compile()` | Configures optimizer, loss, and metrics | No |
| `fit()` on training batches | Runs forward passes, backpropagation, and optimizer steps | Yes |
| Validation inside `fit()` | Measures performance after an epoch | No |
| `evaluate()` | Measures a trained model on a supplied dataset | No |
| `predict()` | Produces model outputs for new inputs | No |

### Key terms

| Term | Meaning |
| --- | --- |
| Optimizer | Algorithm that uses gradients to update parameters |
| Loss | Differentiable error signal minimized during training |
| Metric | Reported measure used to interpret performance |
| Batch | Group of examples used for one parameter update |
| Epoch | One complete pass through the training data |
| Validation set | Development data used to monitor and select models |
| Test set | Untouched data used for final evaluation |
| Convergence | Training has reached a region where the monitored objective changes little |

### Reading common training patterns

| Observation | Likely interpretation | Possible response |
| --- | --- | --- |
| Training and validation loss both decrease | Learning is progressing | Continue while validation improves |
| Training loss decreases, validation loss rises | Overfitting | Stop earlier, regularize, augment, or gather data |
| Both losses remain high | Underfitting or optimization difficulty | Revisit capacity, learning rate, inputs, or training time |
| Loss changes erratically | Learning rate may be high or data may be noisy | Tune optimization and inspect the pipeline |
| Accuracy is high but minority recall is low | Class imbalance is being hidden | Inspect confusion matrix and class-sensitive metrics |

These are diagnostic clues rather than proofs; several causes can produce similar curves.

### Practical checklist

- Confirm that the final sigmoid output, binary labels, and binary cross-entropy agree.
- Check `training_set.class_indices` to understand which class is encoded as 1.
- Inspect `cnn.summary()` before training.
- Track validation loss as well as accuracy.
- Save the best checkpoint rather than assuming the final epoch is best.
- Use early stopping when the correct training duration is unknown.
- Keep a truly untouched test set for the final estimate whenever possible.
- Record the random seed, software versions, preprocessing, and hyperparameters for reproducibility.
- Remember that stochastic initialization, shuffling, and augmentation can change results between runs.

### Quick self-check

1. Does calling `compile()` train the network?
2. Why is binary cross-entropy appropriate for this model?
3. What is the difference between a loss and a metric?
4. Are weights updated while Keras processes `validation_data`?
5. Why should the course's `test_set` more accurately be called a validation set?
6. Does `epochs=25` guarantee that the best model occurs at epoch 25?
7. What information is stored in the object returned by `fit()`?
8. Why can accuracy be misleading on an imbalanced dataset?

### Answers

1. No. It only configures the optimizer, loss, and metrics.
2. The task has binary labels and the model produces one sigmoid probability.
3. The loss supplies the differentiable optimization objective; metrics report interpretable performance measures.
4. No. Validation uses forward passes without backpropagation or optimizer updates.
5. It is inspected every epoch and can influence model-development decisions.
6. No. Twenty-five is only the requested number of passes; an earlier epoch may generalize better.
7. A `History` object containing per-epoch training and validation losses and metrics.
8. A model can achieve high overall accuracy by favoring the majority class while performing poorly on rare examples.

### Final takeaway

Compiling defines how the CNN will learn; fitting performs the learning. Training batches update the parameters, while validation batches provide an independent check after each epoch. Reliable model development monitors validation behavior, retains the best checkpoint, and postpones final test evaluation until all choices are complete.


## Making a Prediction on a Single Image

After training, the CNN can classify an image it has not seen before. This is called **inference**. The central rule is that an inference image must follow the same input contract used during training: the same color channels, spatial size, numerical scale, and batch structure.

A single-image prediction consists of six steps:

1. load the image;
2. resize it to the model's expected dimensions;
3. convert it to a numerical array;
4. apply the same deterministic preprocessing used during training;
5. add a batch dimension; and
6. run the model and decode the returned probability.

### 1. Importing the required tools

The course-era code imports NumPy and the TensorFlow Keras image module:

```python
import numpy as np
from tensorflow.keras.preprocessing import image
```

Current Keras exposes the corresponding loading utilities through `keras.utils`:

```python
import numpy as np
import keras
```

Both styles express the same basic workflow. The remaining examples first explain the notebook's API, then provide a compact current-style version.

### 2. Loading and resizing the image

```python
test_image = image.load_img(
    'dataset/single_prediction/cat_or_dog_1.jpg',
    target_size=(64, 64),
)
```

`load_img` reads the file as a PIL image. `target_size=(64, 64)` resizes it to the height and width expected by the CNN. Because the default color mode is RGB, the loaded image has three color channels, matching the model input shape `(64, 64, 3)`.

Resizing is compulsory for this fixed-input model, but it can distort an image when its aspect ratio differs substantially from 1:1. In a production pipeline, cropping, padding, or aspect-ratio-preserving resizing may be preferable if it matches the training procedure.

### 3. Converting the image to an array

```python
test_image = image.img_to_array(test_image)
```

For the default channels-last format, the result is a rank-3 NumPy array with shape:

```text
(64, 64, 3)
```

This is one image, not yet a batch. The model does not expect a two-dimensional array; a `Conv2D` model normally receives a rank-4 tensor shaped `(batch, height, width, channels)`.

### 4. Applying the same rescaling used during training

The training and validation generators in this notebook use `rescale=1./255`, so their pixel values lie in `[0, 1]`. The single image must receive the same transformation:

```python
test_image = test_image / 255.0
```

This line is missing from the notebook's original prediction cell. Without it, the model receives values in approximately `[0, 255]` even though it learned from values in `[0, 1]`. That train–inference mismatch can make predictions unreliable.

If rescaling is instead implemented as the first layer of the model—for example, `tf.keras.layers.Rescaling(1./255)`—raw `[0, 255]` image arrays should be passed to the model and must not be manually divided again. Preprocessing should happen exactly once.

### 5. Adding the batch dimension

```python
test_image = np.expand_dims(test_image, axis=0)
```

This inserts a leading dimension, changing the shape from:

```text
(64, 64, 3) -> (1, 64, 64, 3)
```

The first dimension is the batch size. Here the batch contains one image. `axis=0` does not create a hierarchy of multiple batches; it creates one batch containing one sample.

### 6. Predicting and decoding the probability

```python
result = cnn.predict(test_image, verbose=0)
probability_of_class_1 = float(result[0, 0])
```

Because the model has one sigmoid output, `predict()` returns an array with shape `(1, 1)`. `result[0, 0]` selects the single output for the single input image. This output is normally a continuous probability such as `0.83`, not a hard value of exactly 0 or 1.

The notebook's inferred class mapping should be inspected explicitly:

```python
training_set.class_indices
# Expected for these directory names: {'cats': 0, 'dogs': 1}
```

If dogs are encoded as 1, a default threshold of 0.5 can be used:

```python
prediction = (
    'dog' if probability_of_class_1 >= 0.5 else 'cat'
)
print(prediction)
```

Comparing `result[0][0] == 1` is generally incorrect because sigmoid outputs are rarely exactly 1. A probability such as `0.93` strongly favors class 1 but would fail an equality check and be incorrectly decoded as class 0.

A threshold of 0.5 is the conventional starting point, not a universal law. In applications where false positives and false negatives have different costs, choose the threshold using validation data and an appropriate metric.

### Corrected course-style prediction

```python
import numpy as np
from tensorflow.keras.preprocessing import image

test_image = image.load_img(
    'dataset/single_prediction/cat_or_dog_1.jpg',
    target_size=(64, 64),
)
test_image = image.img_to_array(test_image)
test_image = test_image / 255.0
test_image = np.expand_dims(test_image, axis=0)

result = cnn.predict(test_image, verbose=0)
probability_of_dog = float(result[0, 0])
prediction = 'dog' if probability_of_dog >= 0.5 else 'cat'

print(f'Prediction: {prediction}')
print(f'Probability of dog: {probability_of_dog:.3f}')
```

The variable name `probability_of_dog` is valid only after confirming that dog is class 1. A more general implementation should derive the names from the saved class mapping rather than hard-coding them.

### Current Keras utility style

```python
import numpy as np
import keras

img = keras.utils.load_img(
    'dataset/single_prediction/cat_or_dog_1.jpg',
    target_size=(64, 64),
)
x = keras.utils.img_to_array(img)
x = x / 255.0
x = np.expand_dims(x, axis=0)

probability_of_dog = float(cnn.predict(x, verbose=0)[0, 0])
prediction = 'dog' if probability_of_dog >= 0.5 else 'cat'
```

For a real deployment, save and load the trained model together with its preprocessing rules, input shape, class names, and decision threshold. A prediction function is only reliable when this full inference contract is preserved.


## Study Notes: Single-Image Inference

### Inference pipeline

$$
\text{File} \rightarrow \text{Load} \rightarrow \text{Resize} \rightarrow \text{Array} \rightarrow \text{Rescale} \rightarrow \text{Batch} \rightarrow \text{Predict} \rightarrow \text{Decode}
$$

For this notebook, the shape and scale progression is:

| Stage | Shape | Typical value range |
| --- | --- | --- |
| PIL image after resize | 64 x 64 RGB | 0 to 255 |
| NumPy image array | `(64, 64, 3)` | 0 to 255 |
| Rescaled image | `(64, 64, 3)` | 0 to 1 |
| One-image batch | `(1, 64, 64, 3)` | 0 to 1 |
| Sigmoid prediction | `(1, 1)` | 0 to 1 |

### Probability interpretation

Assuming the mapping is `cat -> 0` and `dog -> 1`:

| Sigmoid output | Default decoded class | Interpretation |
| ---: | --- | --- |
| 0.05 | Cat | Low estimated probability of dog |
| 0.49 | Cat | Just below the 0.5 decision threshold |
| 0.50 | Dog with `>= 0.5` | Exactly at the chosen threshold |
| 0.91 | Dog | High estimated probability of dog |

A sigmoid output is a model score interpreted as a probability under the training setup. It is not a guarantee that 90% of equally scored real-world images are dogs; that stronger claim requires checking probability calibration on representative held-out data.

### Key distinctions

| Concept | Meaning |
| --- | --- |
| Inference | Running a trained model without updating its weights |
| Preprocessing contract | Exact shape, scale, channel order, and transformations expected by the model |
| Batch dimension | Leading axis that groups one or more samples |
| Probability/score | Continuous sigmoid output |
| Threshold | Rule that converts a score into a class decision |
| Class mapping | Association between numeric labels and human-readable class names |

### Common mistakes

- Forgetting to rescale a prediction image when rescaling occurred outside the model during training.
- Rescaling twice when the model already contains a `Rescaling` layer.
- Passing `(64, 64, 3)` instead of a batch shaped `(1, 64, 64, 3)`.
- Comparing a sigmoid probability directly with `0` or `1` instead of applying a threshold.
- Assuming the class-to-index mapping rather than inspecting or saving it.
- Reversing height and width for non-square model inputs.
- Changing RGB to BGR, grayscale, or another channel convention at inference time.
- Reporting a class without retaining the associated score and model version.
- Treating a notebook prediction as a complete production deployment.

### Production checklist

- Save the trained model or best checkpoint.
- Save the class names in their numeric order.
- Keep preprocessing with the model or version it alongside the model.
- Validate file type, color mode, and input dimensions.
- Select the decision threshold using validation data when costs are asymmetric.
- Return the model score as well as the decoded class where appropriate.
- Test malformed, unexpected, and out-of-distribution inputs.
- Monitor real-world performance and data drift after deployment.

### Quick self-check

1. Why must the single image be resized to 64 by 64?
2. Why must this notebook's inference image be divided by 255?
3. What shape does `img_to_array` produce for this RGB image?
4. What shape is passed to the CNN after `expand_dims(axis=0)`?
5. What does `result[0, 0]` select?
6. Why is `result[0, 0] == 1` an unsafe classification rule?
7. How do you determine what the sigmoid probability represents?
8. When might a threshold other than 0.5 be preferable?

### Answers

1. The trained model expects the same spatial input shape used to build and train it.
2. Its generators trained the model on pixel values scaled to `[0, 1]`; inference must match that scale.
3. `(64, 64, 3)`.
4. `(1, 64, 64, 3)`.
5. The only output unit for the first and only image in the batch.
6. Sigmoid normally returns a continuous value such as 0.87 rather than exactly 1.
7. Inspect and preserve the class-to-index mapping used when the training labels were created.
8. When false positives and false negatives have different costs or validation metrics favor another operating point.

### Final takeaway

Single-image prediction is straightforward only when inference reproduces the training input pipeline exactly. Resize, rescale, add the batch dimension, interpret the sigmoid as a continuous score, and decode it using a verified class mapping and an appropriate threshold.
